<a href="https://colab.research.google.com/github/GoncalvesAlessandro/pocs-arquiteturas-busca/blob/main/02_ground_truth_avaliacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Avaliação de relevância e métricas

Este notebook consome os arquivos produzidos pelo notebook 01, constrói ou atualiza o Ground Truth por meio de julgamento automatizado e calcula Precision, Recall e F1-score.

O tempo de execução utilizado nas métricas é a mediana das cinco repetições oficiais realizada no Pipe 1. Os tempos de preparação/indexação permanecem separados.

Julgamentos existentes só são reutilizados quando o documento, o texto avaliado, a intenção, o modelo, o limite de caracteres e o prompt correspondem à entrada anterior. Documentos removidos do pool atual são descartados da versão atual do Ground Truth.


## 1. Instalar bibliotecas

In [ ]:
!pip -q install --upgrade openai pandas numpy tqdm

## 2. Conectar Drive e configurar a OpenAI API

No Google Colab, abra **Secrets** (ícone de chave) e crie `OPENAI_API_KEY`. Habilite o acesso desse notebook ao segredo.

O modelo padrão abaixo é `gpt-6-luna`. Se sua conta/projeto não tiver acesso, substitua por um modelo disponível no seu painel/API. A API é cobrada separadamente da assinatura ChatGPT.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import json, time, math, hashlib, shutil, statistics
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/TCC_Base_Dados_POC')
INPUT_DIR = BASE_DIR / 'resultados_experimentos'
OUTPUT_DIR = INPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = INPUT_DIR / 'resultados_busca.json'
POOL_PATH = INPUT_DIR / 'pool_candidatos.json'
PREVIOUS_POOL_PATH = INPUT_DIR / 'pool_candidatos_anterior.json'
INDEX_TIMES_PATH = INPUT_DIR / 'tempos_indexacao.json'
GROUND_TRUTH_PATH = OUTPUT_DIR / 'ground_truth.json'

OPENAI_MODEL = 'gpt-6-luna'
MAX_DOC_CHARS = 7000
BATCH_SIZE = 5

api_key = userdata.get('OPENAI_API_KEY')
if not api_key:
    raise RuntimeError('Não encontrei OPENAI_API_KEY em Colab > Secrets.')

client = OpenAI(api_key=api_key)

for path in [RESULTS_PATH, POOL_PATH, INDEX_TIMES_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f'Arquivo não encontrado: {path}\n'
            'Execute primeiro o notebook 01 até o final.'
        )

print('Configuração concluída. Modelo avaliador:', OPENAI_MODEL)


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Mounted at /content/drive
Configuração concluída. Modelo avaliador: gpt-6-luna


## 3. Carregar os resultados e o pool

O notebook lê os documentos candidatos já preparados. Não carrega nem reexecuta BM25, Booleano ou Sentence Transformers.

In [ ]:
with open(RESULTS_PATH, 'r', encoding='utf-8') as f:
    resultados_busca = json.load(f)

with open(POOL_PATH, 'r', encoding='utf-8') as f:
    pool_data = json.load(f)

with open(INDEX_TIMES_PATH, 'r', encoding='utf-8') as f:
    tempos_indexacao = json.load(f)

if len(resultados_busca) != 30:
    raise ValueError(
        f'Esperadas 30 execuções (10 intenções x 3 arquiteturas), '
        f'encontradas {len(resultados_busca)}.'
    )

# Confere se todas as combinações esperadas aparecem exatamente uma vez.
chaves_execucao = [
    (r['base'], r['query_id'], r['arquitetura'])
    for r in resultados_busca
]
if len(set(chaves_execucao)) != 30:
    raise ValueError('Há combinações de base/consulta/arquitetura duplicadas.')

arquiteturas_esperadas = {'booleano', 'bm25', 'vetorial'}
chaves_esperadas = {
    (base_name, query_id, arquitetura)
    for base_name, queries in pool_data['bases'].items()
    for query_id in queries
    for arquitetura in arquiteturas_esperadas
}
if set(chaves_execucao) != chaves_esperadas:
    faltantes = chaves_esperadas - set(chaves_execucao)
    inesperadas = set(chaves_execucao) - chaves_esperadas
    raise ValueError(
        f'Combinações inconsistentes. Ausentes: {sorted(faltantes)}; '
        f'inesperadas: {sorted(inesperadas)}.'
    )

pool_por_consulta = {}
for base_name, queries in pool_data['bases'].items():
    for query_id, pool_item in queries.items():
        pool_por_consulta[(base_name, query_id)] = {
            d['document_id'] for d in pool_item['documents']
        }
        print(base_name, query_id, len(pool_item['documents']), 'documentos no pool')

for result in resultados_busca:
    ids = result['document_ids']
    chave = (result['base'], result['query_id'])
    if chave not in pool_por_consulta:
        raise ValueError(f'Pool ausente para {chave}.')
    fora_do_pool = set(ids) - pool_por_consulta[chave]
    if fora_do_pool:
        raise ValueError(
            f'{len(fora_do_pool)} documentos recuperados não estão no pool de {chave}. '
            'Não prossiga para as métricas.'
        )

    tempos = result.get('latencias_repeticoes_segundos', [])
    if len(tempos) != 5 or result.get('repeticoes_medicao') != 5:
        raise ValueError(
            f'Esperadas cinco medições oficiais em {chave}/{result["arquitetura"]}.'
        )
    if result.get('aquecimentos') != 1:
        raise ValueError(
            f'Esperado um aquecimento em {chave}/{result["arquitetura"]}.'
        )
    if any((not math.isfinite(float(t)) or float(t) < 0) for t in tempos):
        raise ValueError(f'Tempo inválido em {chave}/{result["arquitetura"]}.')

    mediana_calculada = statistics.median(float(t) for t in tempos)
    if not math.isclose(
        float(result['latencia_segundos']),
        mediana_calculada,
        rel_tol=1e-9,
        abs_tol=1e-12
    ):
        raise ValueError(
            f'A latência registrada não corresponde à mediana das repetições em '
            f'{chave}/{result["arquitetura"]}.'
        )

print('Execuções carregadas:', len(resultados_busca))
print('Medições oficiais:', sum(len(r['latencias_repeticoes_segundos']) for r in resultados_busca))
print('Protocolo informado:', tempos_indexacao.get('protocolo_medicao', 'ausente'))


constituicao_federal Q1 122 documentos no pool
constituicao_federal Q2 142 documentos no pool
constituicao_federal Q3 251 documentos no pool
constituicao_federal Q4 123 documentos no pool
constituicao_federal Q5 121 documentos no pool
wikipedia Q1 120 documentos no pool
wikipedia Q2 123 documentos no pool
wikipedia Q3 478 documentos no pool
wikipedia Q4 115 documentos no pool
wikipedia Q5 116 documentos no pool
Execuções carregadas: 30
Medições oficiais: 150
Protocolo informado: {'aquecimentos_por_consulta': 1, 'repeticoes_oficiais_por_consulta': 5, 'agregacao_latencia': 'mediana', 'sincronizacao_cuda': 'antes e depois das medições vetoriais e da geração dos embeddings, quando CUDA estiver disponível'}


## 4. Definir o protocolo de julgamento

O avaliador recebe apenas:
- a base de origem;
- a intenção de pesquisa;
- o texto do documento e seu identificador.

Não recebe o nome da arquitetura, score ou posição do documento. Assim, o julgamento não é condicionado pelo sistema que recuperou o documento.

Critérios:
- `1`: o documento responde diretamente ou contribui materialmente para a intenção;
- `0`: o documento não contribui de forma significativa, mesmo que compartilhe palavras.

A justificativa e a confiança são registradas para auditoria. A confiança não substitui validação humana.

In [ ]:
JUDGE_SCHEMA = {
    'type': 'object',
    'properties': {
        'judgments': {
            'type': 'array',
            'items': {
                'type': 'object',
                'properties': {
                    'document_id': {'type': 'string'},
                    'relevance': {'type': 'integer', 'enum': [0, 1]},
                    'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
                    'justification': {'type': 'string'}
                },
                'required': ['document_id', 'relevance', 'confidence', 'justification'],
                'additionalProperties': False
            }
        }
    },
    'required': ['judgments'],
    'additionalProperties': False
}

JUDGE_INSTRUCTIONS = '''
Você é um avaliador de relevância para um experimento de recuperação da informação.
Classifique cada documento em relação à INTENÇÃO DE PESQUISA, não à forma literal da consulta.

Relevância 1: o conteúdo responde diretamente ou contribui materialmente para a intenção.
Relevância 0: o documento apenas compartilha palavras, trata de assunto periférico ou não contribui.
Não presuma informações que não estejam no documento. Avalie apenas o texto fornecido.
Considere o contexto da base. Seja consistente entre documentos.
Não use a origem, algoritmo, posição ou score do documento.
Retorne exatamente um julgamento para cada document_id recebido.
'''

def classificar_lote(intencao, base, documentos):
    entrada = {
        'base': base,
        'intencao_de_pesquisa': intencao,
        'documentos': [
            {
                'document_id': d['document_id'],
                'texto': d['texto'][:MAX_DOC_CHARS]
            }
            for d in documentos
        ]
    }

    response = client.responses.create(
        model=OPENAI_MODEL,
        instructions=JUDGE_INSTRUCTIONS,
        input=json.dumps(entrada, ensure_ascii=False),
        text={
            'format': {
                'type': 'json_schema',
                'name': 'relevance_judgments',
                'strict': True,
                'schema': JUDGE_SCHEMA
            }
        },
        store=False
    )

    if not response.output_text:
        raise RuntimeError('A API não retornou JSON de classificação.')
    return json.loads(response.output_text)['judgments']

## 5. Gerar Ground Truth com salvamento incremental

O progresso é salvo após cada lote. Se a execução for interrompida, ao rodar novamente o notebook, documentos já julgados não serão enviados novamente à API.

**Custo:** cada documento do pool, por consulta, gera um julgamento de API. Antes de executar, confira o número de documentos no pool e o modelo escolhido.

In [ ]:
def fingerprint_entrada(base_name, query_id, intencao, documento):
    """Identifica a entrada exata usada no julgamento de um documento."""
    payload = {
        'base': base_name,
        'query_id': query_id,
        'intencao': intencao,
        'document_id': documento['document_id'],
        'texto_avaliado': documento['texto'][:MAX_DOC_CHARS],
        'modelo': OPENAI_MODEL,
        'max_document_chars': MAX_DOC_CHARS,
        'prompt': JUDGE_INSTRUCTIONS
    }
    serialized = json.dumps(payload, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(serialized.encode('utf-8')).hexdigest()


# Carrega julgamentos anteriores sem assumir que todos ainda são válidos.
existing = None
previous_pool = {}
if GROUND_TRUTH_PATH.exists():
    with open(GROUND_TRUTH_PATH, 'r', encoding='utf-8') as f:
        existing = json.load(f)

if PREVIOUS_POOL_PATH.exists():
    with open(PREVIOUS_POOL_PATH, 'r', encoding='utf-8') as f:
        previous_pool = json.load(f)
    print('Snapshot do pool anterior encontrado; será usado para validar julgamentos reaproveitados.')

old_meta = (existing or {}).get('metadata', {})
config_antiga_compativel = (
    existing is not None
    and old_meta.get('modelo') == OPENAI_MODEL
    and old_meta.get('max_document_chars') == MAX_DOC_CHARS
)

if existing is not None and not config_antiga_compativel:
    backup_gt = OUTPUT_DIR / 'ground_truth_anterior.json'
    shutil.copy2(GROUND_TRUTH_PATH, backup_gt)
    print('Configuração do Ground Truth anterior difere; arquivo preservado em:', backup_gt)

# Índice do pool anterior: documento e intenção utilizados no julgamento antigo.
previous_docs_by_query = {}
for old_base, old_queries in previous_pool.get('bases', {}).items():
    for old_query_id, old_item in old_queries.items():
        previous_docs_by_query[(old_base, old_query_id)] = {
            d['document_id']: d for d in old_item.get('documents', [])
        }

old_judgments_by_query = {}
if config_antiga_compativel:
    for old_base, old_queries in existing.get('bases', {}).items():
        for old_query_id, judgments in old_queries.items():
            old_judgments_by_query[(old_base, old_query_id)] = {
                item['document_id']: item for item in judgments
            }

prompt_hash = hashlib.sha256(JUDGE_INSTRUCTIONS.encode('utf-8')).hexdigest()
ground_truth = {
    'metadata': {
        'avaliador': 'OpenAI API',
        'modelo': OPENAI_MODEL,
        'metodo': 'LLM-as-a-judge; julgamento binário de relevância',
        'escala': {'1': 'relevante', '0': 'não relevante'},
        'max_document_chars': MAX_DOC_CHARS,
        'batch_size': BATCH_SIZE,
        'prompt_sha256': prompt_hash,
        'fingerprint_version': 1,
        'nota': 'Julgamentos automatizados; não validados por avaliadores humanos. Julgamentos antigos só são reutilizados quando a entrada avaliada é idêntica.'
    },
    'bases': {}
}

for base_name, queries in pool_data['bases'].items():
    ground_truth['bases'].setdefault(base_name, {})

    for query_id, pool_item in queries.items():
        current_pool_docs = pool_item['documents']
        current_by_id = {d['document_id']: d for d in current_pool_docs}
        current_fingerprints = {
            doc_id: fingerprint_entrada(
                base_name, query_id, pool_item['intencao'], documento
            )
            for doc_id, documento in current_by_id.items()
        }

        old_judgments = old_judgments_by_query.get((base_name, query_id), {})
        old_pool_docs = previous_docs_by_query.get((base_name, query_id), {})

        # Só mantém julgamentos cuja entrada atual seja igual à entrada antiga.
        current = {}
        for doc_id, judgment in old_judgments.items():
            if doc_id not in current_by_id:
                continue  # documento não pertence mais ao pool atual

            old_fingerprint = judgment.get('_input_fingerprint')
            if not old_fingerprint and doc_id in old_pool_docs:
                old_item = old_pool_docs[doc_id]
                old_intencao = previous_pool['bases'][base_name][query_id]['intencao']
                old_fingerprint = fingerprint_entrada(
                    base_name, query_id, old_intencao, old_item
                )

            if old_fingerprint == current_fingerprints[doc_id]:
                reused = dict(judgment)
                reused['_input_fingerprint'] = current_fingerprints[doc_id]
                current[doc_id] = reused

        pending = [
            d for d in current_pool_docs
            if d['document_id'] not in current
        ]

        reused_count = len(current)
        print(
            f'{base_name}/{query_id}: {reused_count} julgamentos válidos reutilizados; '
            f'{len(pending)} novos/alterados para avaliar.'
        )

        for start in tqdm(
            range(0, len(pending), BATCH_SIZE),
            desc=f'{base_name}/{query_id}'
        ):
            batch = pending[start:start + BATCH_SIZE]
            attempts = 0

            while True:
                try:
                    judgments = classificar_lote(
                        pool_item['intencao'],
                        base_name,
                        batch
                    )
                    break
                except Exception as error:
                    attempts += 1
                    if attempts >= 5:
                        raise
                    wait = min(2 ** attempts, 30)
                    print(f'Erro na API: {error}. Tentando novamente em {wait}s.')
                    time.sleep(wait)

            expected_ids = [d['document_id'] for d in batch]
            returned_ids = [j['document_id'] for j in judgments]

            if (
                len(returned_ids) != len(expected_ids)
                or len(set(returned_ids)) != len(returned_ids)
                or set(expected_ids) != set(returned_ids)
            ):
                raise ValueError(
                    f'IDs retornados ausentes, duplicados ou inesperados: '
                    f'{base_name}/{query_id}'
                )

            for judgment in judgments:
                doc_id = judgment['document_id']
                judgment['_input_fingerprint'] = current_fingerprints[doc_id]
                current[doc_id] = judgment

            # Salva somente documentos do pool atual e na ordem do pool.
            ground_truth['bases'][base_name][query_id] = [
                current[d['document_id']]
                for d in current_pool_docs
                if d['document_id'] in current
            ]
            with open(GROUND_TRUTH_PATH, 'w', encoding='utf-8') as f:
                json.dump(ground_truth, f, ensure_ascii=False, indent=2)

        # Também grava a consulta quando todos os julgamentos foram reaproveitados.
        ground_truth['bases'][base_name][query_id] = [
            current[d['document_id']]
            for d in current_pool_docs
            if d['document_id'] in current
        ]
        with open(GROUND_TRUTH_PATH, 'w', encoding='utf-8') as f:
            json.dump(ground_truth, f, ensure_ascii=False, indent=2)

print('Ground Truth salvo em:', GROUND_TRUTH_PATH)


Snapshot do pool anterior encontrado; será usado para validar julgamentos reaproveitados.
constituicao_federal/Q1: 0 julgamentos válidos reutilizados; 122 novos/alterados para avaliar.


constituicao_federal/Q1:   0%|          | 0/25 [00:00<?, ?it/s]

constituicao_federal/Q2: 142 julgamentos válidos reutilizados; 0 novos/alterados para avaliar.


constituicao_federal/Q2: 0it [00:00, ?it/s]

constituicao_federal/Q3: 250 julgamentos válidos reutilizados; 1 novos/alterados para avaliar.


constituicao_federal/Q3:   0%|          | 0/1 [00:00<?, ?it/s]

... (11 outputs left out)

wikipedia/Q4: 0it [00:00, ?it/s]

wikipedia/Q5: 0 julgamentos válidos reutilizados; 116 novos/alterados para avaliar.


wikipedia/Q5:   0%|          | 0/24 [00:00<?, ?it/s]

Ground Truth salvo em: /content/drive/MyDrive/TCC_Base_Dados_POC/resultados_experimentos/ground_truth.json


## 6. Calcular as métricas definidas

As métricas de relevância são calculadas para as três arquiteturas:
- **Precision:** proporção de documentos recuperados que foram julgados relevantes.
- **Recall:** proporção dos documentos relevantes julgados no pool que foi recuperada.
- **F1-score:** média harmônica entre Precision e Recall.
- **Tempo de execução:** latência registrada na Pipeline 1.

Precision, Recall e F1-score usam as mesmas fórmulas para Booleano, BM25 e Vetorial. O Recall é relativo ao pool de documentos julgados pelo LLM, não ao corpus inteiro.

A `latencia_segundos` recebida da Pipeline 1 representa a **mediana das cinco repetições oficiais**, após uma execução de aquecimento descartada. A média consolidada no final desta pipeline é a média aritmética simples das cinco latências medianas de cada arquitetura/base.

**Atenção à interpretação:** a busca booleana retorna todos os documentos que satisfazem suas condições; BM25 e Vetorial retornam até 50 documentos. Não calculamos MRR ou nDCG.


In [ ]:

def calcular_metricas(retrieved_ids, qrels):
    """Calcula Precision, Recall e F1 sobre o conjunto efetivamente recuperado.

    qrels: dict {document_id: 0 ou 1}, referente aos documentos julgados no pool.
    """
    retrieved_set = set(retrieved_ids)

    relevant_in_pool = {
        doc_id for doc_id, relevance in qrels.items()
        if int(relevance) == 1
    }

    relevant_retrieved = len(retrieved_set & relevant_in_pool)

    precision = (
        relevant_retrieved / len(retrieved_set)
        if retrieved_set else 0.0
    )

    recall = (
        relevant_retrieved / len(relevant_in_pool)
        if relevant_in_pool else 0.0
    )

    f1_score = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) else 0.0
    )

    return {
        'precision': precision,
        'recall': recall,
        'f1_score': f1_score,
        'relevantes_no_pool': len(relevant_in_pool),
        'relevantes_recuperados': relevant_retrieved
    }


metric_rows = []

for result in resultados_busca:
    base_name = result['base']
    query_id = result['query_id']
    architecture = result['arquitetura']

    # Carrega os julgamentos do Ground Truth.
    qrels_list = ground_truth['bases'][base_name][query_id]

    # Verifica se existem documentos duplicados no Ground Truth.
    qrels_ids_list = [x['document_id'] for x in qrels_list]

    if len(qrels_ids_list) != len(set(qrels_ids_list)):
        raise ValueError(
            f'Existem documentos duplicados no Ground Truth de '
            f'{base_name}/{query_id}.'
        )

    # Converte os julgamentos para um dicionário.
    qrels = {
        x['document_id']: int(x['relevance'])
        for x in qrels_list
    }

    # Verifica se todos os documentos do pool possuem julgamento.
    pool_ids = pool_por_consulta[(base_name, query_id)]
    qrels_ids = set(qrels)

    if qrels_ids != pool_ids:
        raise ValueError(
            f'Ground Truth incompleto ou inconsistente em {base_name}/{query_id}. '
            f'Ausentes: {len(pool_ids - qrels_ids)}; '
            f'extras: {len(qrels_ids - pool_ids)}.'
        )

    # Recupera os documentos retornados pela arquitetura.
    retrieved_ids = result['document_ids']

    # Verifica se todos os documentos recuperados possuem julgamento.
    unjudged = set(retrieved_ids) - qrels_ids

    if unjudged:
        raise ValueError(
            f'{len(unjudged)} documentos recuperados não possuem julgamento em '
            f'{base_name}/{query_id}/{architecture}.'
        )

    # Calcula Precision, Recall e F1-score.
    metrics = calcular_metricas(retrieved_ids, qrels)

    # Recupera os tempos das cinco medições oficiais.
    repeat_times = result.get('latencias_repeticoes_segundos', [])

    metric_rows.append({
        'base': base_name,
        'query_id': query_id,
        'arquitetura': architecture,

        # A latência registrada na Pipeline 1 é a mediana das cinco repetições.
        'tempo_execucao_segundos': result['latencia_segundos'],

        'repeticoes_tempo': len(repeat_times),
        'tempos_repeticoes_segundos': json.dumps(repeat_times),

        'total_recuperado': result['total_recuperado'],

        **metrics
    })


# Consolida os resultados em um DataFrame.
metricas_df = pd.DataFrame(metric_rows)

display(metricas_df)


,base,query_id,arquitetura,tempo_execucao_segundos,repeticoes_tempo,tempos_repeticoes_segundos,total_recuperado,precision,recall,f1_score,relevantes_no_pool,relevantes_recuperados
0,constituicao_federal,Q1,booleano,0.024451,5,"[0.025379937000025166, 0.02445098600003348, 0....",1,1.000000,0.076923,0.142857,13,1
1,constituicao_federal,Q1,bm25,0.006285,5,"[0.006789315999981227, 0.006367907000026207, 0...",50,0.200000,0.769231,0.317460,13,10
2,constituicao_federal,Q1,vetorial,0.011397,5,"[0.01334878199998002, 0.013157809999938763, 0....",50,0.120000,0.461538,0.190476,13,6
3,constituicao_federal,Q2,booleano,0.028455,5,"[0.028228216999991673, 0.028455027000063637, 0...",31,0.870968,0.473684,0.613636,57,27
4,constituicao_federal,Q2,bm25,0.008510,5,"[0.008727399000008518, 0.008513923999998951, 0...",50,0.720000,0.631579,0.672897,57,36
5,constituicao_federal,Q2,vetorial,0.011362,5,"[0.011599810000006983, 0.011361825000108183, 0...",50,0.260000,0.228070,0.242991,57,13
6,constituicao_federal,Q3,booleano,0.028248,5,"[0.02909684600001583, 0.02858253900001273, 0.0...",214,0.990654,0.981481,0.986047,216,212
7,constituicao_federal,Q3,bm25,0.006219,5,"[0.006656318999944233, 0.006219084999997904, 0...",50,0.980000,0.226852,0.368421,216,49
8,constituicao_federal,Q3,vetorial,0.011175,5,"[0.010949916999948073, 0.011175417999993442, 0...",50,0.940000,0.217593,0.353383,216,47
9,constituicao_federal,Q4,booleano,0.030241,5,"[0.030429876000084732, 0.03220931099997415, 0....",8,0.750000,0.230769,0.352941,26,6


## 7. Exportar métricas e resumo comparativo

In [ ]:
metrics_path = OUTPUT_DIR / 'metricas_por_consulta.csv'
summary_path = OUTPUT_DIR / 'resumo_comparativo.csv'

metricas_df.to_csv(metrics_path, index=False, encoding='utf-8-sig')

# Médias por base e arquitetura. Para latência, é a média das cinco medianas por consulta.
summary = (
    metricas_df.groupby(['base', 'arquitetura'], dropna=False)
    .agg({
        'precision': 'mean',
        'recall': 'mean',
        'f1_score': 'mean',
        'tempo_execucao_segundos': 'mean',
        'total_recuperado': 'mean'
    })
    .reset_index()
)
summary.to_csv(summary_path, index=False, encoding='utf-8-sig')

print('Arquivos exportados:')
for path in [GROUND_TRUTH_PATH, metrics_path, summary_path]:
    print(path.name, f'{path.stat().st_size:,} bytes')

display(summary)


Arquivos exportados:
ground_truth.json 633,382 bytes
metricas_por_consulta.csv 6,415 bytes
resumo_comparativo.csv 701 bytes


,base,arquitetura,precision,recall,f1_score,tempo_execucao_segundos,total_recuperado
0,constituicao_federal,bm25,0.496000,0.525080,0.417871,0.007382,50.0
1,constituicao_federal,booleano,0.922324,0.358454,0.430525,0.028427,51.0
2,constituicao_federal,vetorial,0.424000,0.445694,0.355866,0.011725,50.0
3,wikipedia,bm25,0.380000,0.567258,0.329215,0.151569,50.0
4,wikipedia,booleano,0.634586,0.360368,0.373291,1.209539,85.0
5,wikipedia,vetorial,0.340000,0.519187,0.262925,0.014588,50.0


## 8. Interpretação metodológica e limitações

1. O Ground Truth foi gerado automaticamente por um LLM. No TCC, descreva-o como **julgamentos automatizados de relevância**, não como verdade absoluta ou avaliação humana.
2. O pool inclui todos os resultados booleanos, até 50 resultados de BM25, até 50 resultados vetoriais e uma amostra aleatória adicional. O Recall é relativo aos documentos julgados no pool, não ao universo completo da base.
3. Precision, Recall e F1-score são calculados com as mesmas fórmulas para as três arquiteturas, considerando o conjunto efetivamente recuperado por cada uma.
4. A busca booleana retorna todos os documentos que satisfazem suas condições, enquanto BM25 e Vetorial retornam até 50 documentos. Essa diferença no volume de resultados deve ser considerada ao interpretar as métricas.
5. O tempo de execução corresponde à latência de cada consulta registrada na Pipeline 1. Os tempos de indexação/preparação permanecem registrados separadamente em `tempos_indexacao.json` e não são somados à latência da consulta.
6. O modelo avaliador, prompt, limite de caracteres e versão do arquivo de consultas devem ser registrados no relatório.
7. A comparação entre algoritmos é feita sobre intenções equivalentes, com sintaxe de consulta específica por arquitetura.
8. O resultado do LLM pode conter erros sistemáticos. Sem validação humana, essa limitação deve ser explicitada no trabalho.


## 9. Conferências antes de finalizar

- Confirme que existem 30 combinações de execução (5 consultas × 2 bases × 3 arquiteturas).
- Confirme que cada combinação possui cinco tempos oficiais e uma latência mediana.
- Confirme que todos os documentos recuperados possuem julgamento no pool atual.
- A média consolidada de latência é calculada sobre as cinco medianas das consultas de cada base/arquitetura.
- O arquivo `pool_candidatos_anterior.json` é usado apenas para validar a reutilização de julgamentos; não é utilizado no cálculo das métricas atuais.
